In [ ]:
import os
import MMCoT
import sys
import json
import numpy as np
import torch
import subprocess
from tqdm import tqdm
from dotenv import load_dotenv
from huggingface_hub import login
from aggregators.aggregator import CookingAggregator

def install_packages():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "sentence-transformers"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "accelerate"])

install_packages()

from sentence_transformers import SentenceTransformer

load_dotenv(dotenv_path=".env")
hf_key = os.getenv("HUGGINGFACE_KEY")

In [ ]:
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_cache = {}

In [ ]:
def get_embedding(text):
    if text not in embedding_cache:
        vec = semantic_model.encode(text)
        embedding_cache[text] = vec / np.linalg.norm(vec)
    return embedding_cache[text]

def semantic_similarity(a, b):
    if not a or not b:
        return 0.0
    vec_a = get_embedding(a)
    vec_b = get_embedding(b)
    return float(np.dot(vec_a, vec_b))

def adaptive_frame_sampling(frame_numbers, base_stride=10, frame_frequency=None):
    if frame_frequency is not None:
        return frame_numbers[::frame_frequency]
    return frame_numbers[::base_stride]

def sliding_window_captioning_dynamic(
    folder_path, frame_numbers, processor, model, model_id,
    window_size=8, base_stride=8, similarity_threshold=0.70
):
    """
    Generates captions for frames using a sliding window approach with dynamic stride adjustment.
    Stride increases when consecutive captions are highly similar, decreases otherwise.
    """
    max_stride = base_stride * 3
    all_captions = []
    prev_caption = None
    stride = base_stride
    idx = 0
    n_frames = len(frame_numbers)

    while idx < n_frames:
        window = frame_numbers[idx:min(idx + window_size, n_frames)]
        print("Window: ", window)
        caption = MMCoT.analyze_sequence_by_indexes(
            folder_path,
            window,
            processor,
            model,
            device=model.device
        )
        print("Caption:", caption)

        if caption is None or str(caption).strip().lower() == "none":
            caption = ""

        if prev_caption and semantic_similarity(prev_caption, caption) >= similarity_threshold:
            stride = min(int(stride * 1.5), max_stride)
        else:
            if caption.strip():
                all_captions.append(caption)
                prev_caption = caption
            stride = base_stride

        idx += max(1, stride)

    return all_captions

def process_scene(
    scene_folder, video_folder, processor, model, model_id,
    window_size=8, base_stride=8, similarity_threshold=0.7, frame_frequency=5
):
    scene_path = os.path.join(video_folder, scene_folder)
    frame_numbers = sorted(
        [f.replace("frame_", "").replace(".jpg", "") for f in os.listdir(scene_path) if f.endswith(".jpg")],
        key=lambda x: int(x)
    )
    frame_numbers = [f"{int(f):04d}" for f in frame_numbers]

    sampled_frames = adaptive_frame_sampling(frame_numbers, base_stride, frame_frequency)

    captions = sliding_window_captioning_dynamic(
        scene_path, sampled_frames, processor, model, model_id,
        window_size=window_size, base_stride=base_stride,
        similarity_threshold=similarity_threshold
    )

    scene_id = int(scene_folder.split("_")[1])
    return scene_id, captions

def process_video(
    video_id, video_folder, processor, model, model_id,
    window_size=10, base_stride=10, similarity_threshold=0.70, frame_frequency=10
):
    scenes = [d for d in os.listdir(video_folder) if os.path.isdir(os.path.join(video_folder, d))]
    output = {video_id: {}}
    for scene_folder in tqdm(scenes, desc=f"Processing scenes for {video_id}"):
        scene_id, captions = process_scene(
            scene_folder, video_folder, processor, model, model_id,
            window_size, base_stride, similarity_threshold, frame_frequency
        )
        output[video_id][scene_id] = captions
    return output

def get_ground_truth(annotations_file, video_id, scene_count):
    with open(annotations_file, "r") as f:
        annotations_data = json.load(f)

    if video_id not in annotations_data["database"]:
        print(f"Warning: {video_id} not found in annotations.")
        return {i: "" for i in range(scene_count)}

    video_annotations = annotations_data["database"][video_id]["annotations"]
    ground_truth_dict = {}
    for i in range(scene_count):
        ground_truth_dict[i] = video_annotations[i]["sentence"] if i < len(video_annotations) else ""
    return ground_truth_dict

def initialize_models(aggregator_model, agg_gpu, cot_gpu):
    aggregator = CookingAggregator(model_id=aggregator_model, device_index=agg_gpu)

    # Load CoT model
    device0 = torch.device(f"cuda:{cot_gpu}")
    processor0, model0 = MMCoT.load_model(model_id="Qwen/Qwen2.5-VL-7B-Instruct", device=device0)
    COT_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

    return aggregator, processor0, model0, COT_MODEL_ID

def run_pipeline(base_folder, sampled_file, output_folder, annotations_file,
                 aggregator, processor0, model0, COT_MODEL_ID):

    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, "validation_results.json")

    final_output = {}
    if os.path.exists(output_path):
        with open(output_path, "r") as f:
            final_output = json.load(f)
        print(f"Loaded existing JSON with {len(final_output)} videos.")

    with open(sampled_file, "r") as f:
        all_videos = [line.strip() for line in f if line.strip()]
    print(f"Found {len(all_videos)} sampled videos to process.")

    for video_rel_path in tqdm(all_videos, desc="Processing sampled videos"):
        _, video_id = video_rel_path.split("/")
        video_folder = os.path.join(base_folder, "validation", video_rel_path)

        if video_id in final_output:
            print(f"Skipping {video_id}... already exists in JSON.")
            continue
        if not os.path.isdir(video_folder):
            print(f"Warning: {video_folder} does not exist. Skipping.")
            continue

        # Scene captions
        video_result = process_video(
            video_id, video_folder, processor0, model0, COT_MODEL_ID,
            window_size=10, base_stride=10, similarity_threshold=0.50, frame_frequency=10
        )

        aggregated_scenes = {}
        scene_ids_sorted = sorted(video_result[video_id].keys())
        for idx, scene_id in enumerate(scene_ids_sorted):
            frame_captions = video_result[video_id][scene_id]
            aggregated_scenes[idx] = aggregator.generate_cooking_summary(frame_captions) if frame_captions else ""

        # Ground truth
        scene_count = len(aggregated_scenes)
        ground_truth_dict = get_ground_truth(annotations_file, video_id, scene_count)

        video_data = {
            idx: {
                "ground_truth": ground_truth_dict.get(idx, ""),
                "predicted": aggregated_scenes.get(idx, "")
            }
            for idx in range(scene_count)
        }

        # Update JSON
        final_output[video_id] = video_data
        with open(output_path, "w") as f:
            json.dump(final_output, f, indent=2)

        del video_result
        torch.cuda.empty_cache()
        print(f"Processed and updated JSON for video: {video_id}")

    print(f"All results saved to {output_path}")

In [ ]:
aggregator_model = "Qwen/Qwen3-4B-Instruct-2507"
cot_gpu = 0
agg_gpu = 1
base_folder = "/workspace/scene_captioner/data/YouCookII/YouCookII/raw_videos"
sampled_file = "sampled_videos.txt"
output_folder = "/workspace/scene_captioner/LLaMACoT/outputs_qwen"
annotations_file = "/workspace/scene_captioner/data/YouCookII/YouCookII/annotations/youcookii_annotations_trainval.json"

if not hf_key:
    raise RuntimeError("HUGGINGFACE_KEY not found in .env file.")
login(token=hf_key)

aggregator, processor0, model0, COT_MODEL_ID = initialize_models(
    aggregator_model=aggregator_model,
    agg_gpu=agg_gpu,
    cot_gpu=cot_gpu
)

In [ ]:
run_pipeline(
    base_folder=base_folder,
    sampled_file=sampled_file,
    output_folder=output_folder,
    annotations_file=annotations_file,
    aggregator=aggregator,
    processor0=processor0,
    model0=model0,
    COT_MODEL_ID=COT_MODEL_ID
)